# Exploração da base de avaliações (Olist)

Objetivo: entender os comentários das avaliações para definir as regras de seleção, limpeza
e metadados da base de conhecimento do RAG. As regras decididas aqui estão implementadas
em `src/preparar_base.py`.

## Leitura dos dados

In [1]:
import pandas as pd
from pathlib import Path

# Dados copiados por src/baixar_dados.py (rode o script antes do notebook)
PASTA_DADOS = Path("../data/raw")   # o notebook fica em notebooks/

# Lê só as tabelas necessárias para a base de conhecimento
avaliacoes = pd.read_csv(PASTA_DADOS / "olist_order_reviews_dataset.csv")
pedidos    = pd.read_csv(PASTA_DADOS / "olist_orders_dataset.csv")
itens      = pd.read_csv(PASTA_DADOS / "olist_order_items_dataset.csv")
produtos   = pd.read_csv(PASTA_DADOS / "olist_products_dataset.csv")

print(avaliacoes.shape)

(99224, 7)


## 1: quantas avaliações têm texto?

In [2]:
# Nulos nas colunas de texto
print(avaliacoes[["review_comment_title", "review_comment_message"]].isna().mean().round(3))

# Avaliações com pelo menos um dos dois textos preenchido
tem_texto = avaliacoes["review_comment_title"].notna() | avaliacoes["review_comment_message"].notna()
print(f"\nCom algum texto: {tem_texto.sum()} de {len(avaliacoes)}")

review_comment_title      0.883
review_comment_message    0.587
dtype: float64

Com algum texto: 42706 de 99224


**Achado:** 42.706 de 99.224 avaliações (43%) têm algum texto. O título falta em 88% dos casos.

**Decisão:** o documento é *título + mensagem*. Avaliações sem texto ficam fora da base, porque não há conteúdo para recuperar.

## 2: os comentários são úteis ou ruído?

In [3]:
comentarios = avaliacoes["review_comment_message"].dropna().str.strip()
tamanho = comentarios.str.len()

print(tamanho.describe().round(1))
print(f"\nCom menos de 10 caracteres: {(tamanho < 10).sum()}")
print(comentarios[tamanho < 10].value_counts().head(15))

count    40977.0
mean        68.4
std         53.8
min          0.0
25%         28.0
50%         53.0
75%         94.0
max        208.0
Name: review_comment_message, dtype: float64

Com menos de 10 caracteres: 3199
review_comment_message
Muito bom    305
Bom          215
Ótimo        158
Recomendo    153
muito bom    144
bom          111
otimo        108
Excelente    108
Otimo        107
Ok            78
recomendo     71
ok            58
Boa           55
.             52
Tudo ok       51
Name: count, dtype: int64


**Achado:** 3.199 comentários têm menos de 10 caracteres, quase todos elogios genéricos ("Bom", "Ótimo", "Ok", "."). O maior tem 208 caracteres.

**Decisão:**
- Descartar textos com menos de 10 caracteres: não dizem *o que* foi bom e competiriam na busca com comentários que citam aspectos concretos.
- Sem chunking: 1 avaliação = 1 documento, porque todo texto cabe com folga no limite do modelo de embedding.

## 3: distribuição de notas entre quem comentou

In [4]:
print(avaliacoes.loc[tem_texto, "review_score"].value_counts(normalize=True).sort_index().round(3))

review_score
1    0.207
2    0.051
3    0.085
4    0.147
5    0.510
Name: proportion, dtype: float64


**Achado:** 51% das avaliações com texto têm nota 5, mas ~26% têm nota 1–2 (≈ 11 mil).

**Decisão:** há volume suficiente de reclamações. A nota entra como metadado para filtrar perguntas sobre clientes insatisfeitos.

## 4: duplicidades

In [5]:
print(f"review_id repetidos: {avaliacoes['review_id'].duplicated().sum()}")
print(f"order_id com mais de uma avaliação: {avaliacoes['order_id'].duplicated().sum()}")

review_id repetidos: 814
order_id com mais de uma avaliação: 551


**Achado:** 814 `review_id` repetidos: o mesmo review replicado em pedidos comprados juntos.

**Decisão:** remover duplicados por `review_id`, que passa a ser o ID único do documento no Chroma.

## 5: o join com a categoria vai multiplicar linhas?

In [6]:
# Quantas categorias distintas existem por pedido
categorias_por_pedido = (
    itens.merge(produtos[["product_id", "product_category_name"]], on="product_id", how="left")
         .groupby("order_id")["product_category_name"].nunique()
)
print(categorias_por_pedido.value_counts().sort_index())

product_category_name
0     1389
1    96550
2      712
3       15
Name: count, dtype: int64


**Achado:** 727 pedidos (~0,7%) têm mais de uma categoria, e 1.389 não têm categoria.

**Decisão:** usar a categoria do item de maior preço (produto principal), o que também evita duplicar o review no join. Pedidos sem categoria recebem `sem_categoria`, porque o Chroma não aceita metadado nulo.

## 6: dá para calcular o atraso?

In [7]:
colunas_data = ["order_purchase_timestamp", "order_delivered_customer_date", "order_estimated_delivery_date"]
print(pedidos[colunas_data].isna().sum())
print(pedidos["order_status"].value_counts())

order_purchase_timestamp            0
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


**Achado:** 2.965 pedidos sem data de entrega (cancelados, em trânsito etc.).

**Decisão:** criar `situacao_entrega` com três valores: `no_prazo`, `atrasado` (entregue depois da data prevista) e `nao_entregue`.

## 7: Validação no notebook

In [8]:
import sys
sys.path.append("../src")
from preparar_base import preparar_base

base = preparar_base()
print(base.shape)
print(base["situacao_entrega"].value_counts())
base.sample(5, random_state=42)

(38624, 8)
situacao_entrega
no_prazo        33105
atrasado         3695
nao_entregue     1824
Name: count, dtype: int64


,review_id,order_id,nota,texto,data_compra,ano_mes,situacao_entrega,categoria
1291,47ebe1dbcd168f20d19a5c4c197f3911,de1f470a407aead094cd885293ea9ba1,1,Comprei um tapete de 2X2.5 e recebi um de 2X1....,2018-02-06,201802,no_prazo,cama_mesa_banho
2279,368345f5a941e8757376276bf39c5b6d,7e52365eb2fa03f8f4ed8a69da1e7fc4,2,comprei 2 toner hp ce278a e só recebi 1 e aind...,2017-03-30,201703,no_prazo,informatica_acessorios
31314,f7ea96cdb986163bb839ec5e77ef1221,98b7e48a869cc004297cb968e5d81401,4,Cumpriu o prometido,2017-11-12,201711,no_prazo,eletronicos
4900,6700be3c2248ebbf8b39ba25e2760761,26cc9c222792e6af792575ab34ee3ecc,5,Produtos de qualidade entregues dentro do prazo,2018-02-08,201802,no_prazo,brinquedos
11020,467e668b373b6d2b4db646614ec66039,7a8adf1b52b8d3c0735699f5708bf513,1,Boa noite. venho por deste comunicar que receb...,2018-02-26,201802,no_prazo,bebes


**Resultado:** base final com **38.624 documentos**: 33.105 entregues no prazo, 3.695 atrasados e 1.824 não entregues. Sem `review_id` duplicado e sem metadado nulo (garantido pelos `assert` em `src/preparar_base.py`).